#Ingesta de carpeta con archivos SCV

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "movie_company"
v_esquema = "movie_silver"
v_tabla = "movies_companies"
v_partition = "file_date"
dbutils.widgets.text("p_esquema", v_esquema)
dbutils.widgets.text("p_tabla", v_tabla)

In [0]:
#1. Leer archivos CSV usando DataFrameReader de Spark

# Define el la estructura personName
movie_company_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("companyId", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
movie_company_df = spark.read\
    .schema(movie_company_schema)\
    .csv(f"{bronze_folder_path}/{v_file_date}/{v_archivo}")

# Mostramos el resultado
display(movie_company_df)

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
movie_company_renamed_df = movie_company_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("companyId", "company_id")

movie_company_renamed_df = add_ingestion_date(movie_company_renamed_df)
movie_company_renamed_df = add_env(movie_company_renamed_df)
movie_company_renamed_df = add_file_date (movie_company_renamed_df)

display(movie_company_renamed_df)


In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
overwrite_partition (v_esquema, v_tabla, v_partition, v_file_date)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
#movie_company_renamed_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_companies")

movie_company_renamed_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {movie_company_renamed_df.count()} registros en la tabla {v_esquema}.{v_tabla}")



In [0]:
dbutils.notebook.exit("El notebook 10. Ingestion folder_movie_company, termino correctamente")